# Metrics Computation for Synthetic Scatterplots

Computes a core set of metrics for the 114,176 synthetic scatterplot cases.

**Input:** `generated_scatterplot_data/cases.csv` + `scatter_points.npz`

**Output:** `generated_scatterplot_data/core/metrics.parquet`

| Group | Metrics | Count |
|---|---|---|
| Correlation / dependence | Pearson r, Spearman ρ, distance correlation | 3 |
| MINE | MIC, MAS, MEV, MCN, MIC − r² | 5 |
| Slopes | endpoint & polyfit × raw/standardised × overall/early/mid/late + segment strength | 20 |
| Bin statistics (equal-width) | amplitude, η², buffer widths (q95−q05), n_valid_bins | 7 |
| X coverage | x_bin_count_cv | 1 |
| LOWESS | residual SD, amplitude, R², sign changes, overall slope | 5 |
| Misc | n_valid | 1 |
| **Total** | | **42** |

In [ ]:
from __future__ import annotations

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr
from statsmodels.nonparametric.smoothers_lowess import lowess

try:
    from minepy import MINE
    HAS_MINEPY = True
except ImportError:
    HAS_MINEPY = False
    print("minepy not installed → MIC / MAS / MEV / MCN will be NaN.")

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings("ignore", category=np.RankWarning)

DATA_DIR = Path("generated_scatterplot_data")

## Load Generated Data

In [ ]:
cases_df = pd.read_csv(DATA_DIR / "cases.csv", low_memory=False)
data = np.load(DATA_DIR / "scatter_points.npz")
x_all = data["x"]
y_all = data["y"]
n_cases, n_points = x_all.shape
print(f"Loaded {n_cases:,} cases × {n_points} points")
print(f"Family distribution: Null={int((cases_df.family_id == 'Null').sum())}, Signal={int((cases_df.family_id != 'Null').sum())}")

## Metric Functions

### Phase 1 — Vectorised (all cases at once)

In [ ]:
def vectorised_pearson(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Row-wise Pearson r.  x, y: (n_cases, n_points)."""
    xc = x - x.mean(axis=1, keepdims=True)
    yc = y - y.mean(axis=1, keepdims=True)
    num = (xc * yc).sum(axis=1)
    den = np.sqrt((xc ** 2).sum(axis=1) * (yc ** 2).sum(axis=1))
    return np.where(den > 0, num / den, np.nan)


def _rank_rows(arr: np.ndarray) -> np.ndarray:
    n_cases, n_points = arr.shape
    ranks = np.empty(arr.shape, dtype=np.float64)
    order = arr.argsort(axis=1)
    rows = np.arange(n_cases)[:, None]
    ranks[rows, order] = np.arange(1, n_points + 1, dtype=np.float64)
    return ranks


def vectorised_spearman(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    return vectorised_pearson(_rank_rows(x), _rank_rows(y))

### Phase 2 — Shared helpers and per-case metric functions

These functions are identical to those in `metrics_computation_full.ipynb`.

In [ ]:
# ── Shared helpers ──

def _to_valid(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    return x[valid], y[valid]


def _safe_div(a, b):
    if b == 0 or not np.isfinite(b):
        return np.nan
    return a / b


def _minmax01(v):
    v = np.asarray(v, dtype=float)
    lo, hi = np.nanmin(v), np.nanmax(v)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return np.full_like(v, np.nan, dtype=float)
    return (v - lo) / (hi - lo)


def _endpoint_slope(x, y):
    if len(x) < 2:
        return np.nan
    return _safe_div(float(y[-1] - y[0]), float(x[-1] - x[0]))


def _polyfit_slope(x, y):
    if len(x) < 3 or np.std(x) == 0:
        return np.nan
    return float(np.polyfit(x, y, 1)[0])


def _segment_masks(n):
    i1, i2 = n // 3, 2 * n // 3
    early  = np.zeros(n, bool); early[:i1]   = True
    middle = np.zeros(n, bool); middle[i1:i2] = True
    late   = np.zeros(n, bool); late[i2:]    = True
    return early, middle, late


def _residual_sd(y, y_hat):
    r = y - y_hat
    return float(np.nanstd(r, ddof=1)) if len(r) >= 2 else np.nan


def _r2(y, y_hat):
    ss_res = float(np.nansum((y - y_hat) ** 2))
    ss_tot = float(np.nansum((y - np.nanmean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan


def _sign_changes(x_curve, y_curve, tol=1e-6):
    x_curve = np.asarray(x_curve, float)
    y_curve = np.asarray(y_curve, float)
    valid = np.isfinite(x_curve) & np.isfinite(y_curve)
    x_curve, y_curve = x_curve[valid], y_curve[valid]
    if len(x_curve) < 4:
        return np.nan
    order = np.argsort(x_curve)
    x_curve, y_curve = x_curve[order], y_curve[order]
    dx = np.diff(x_curve)
    dy = np.diff(y_curve)
    ok = dx != 0
    if ok.sum() < 3:
        return np.nan
    slopes = dy[ok] / dx[ok]
    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > tol] = 1
    signs[slopes < -tol] = -1
    nz = signs[signs != 0]
    return float(np.sum(nz[1:] != nz[:-1])) if len(nz) >= 2 else 0.0


def _make_bins(x, n_bins=10, bin_type="equal_width"):
    if bin_type == "equal_width":
        return np.asarray(
            pd.cut(x, bins=n_bins, labels=False, include_lowest=True, duplicates="drop"),
            dtype=float,
        )
    elif bin_type == "equal_count":
        return np.asarray(
            pd.qcut(x, q=n_bins, labels=False, duplicates="drop"),
            dtype=float,
        )
    raise ValueError(f"Unknown bin_type: {bin_type}")

In [ ]:
# ── Distance ──

def _double_center(a):
    a = a.reshape(-1, 1)
    dist = squareform(pdist(a))
    return dist - dist.mean(axis=0, keepdims=True) - dist.mean(axis=1, keepdims=True) + dist.mean()


def _distance_metrics(x, y):
    r = {"distance_covariance": np.nan, "distance_correlation": np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return r
    A = _double_center(x)
    B = _double_center(y)
    dcov2_xy = (A * B).mean()
    dcov_xy = float(np.sqrt(max(dcov2_xy, 0)))
    dcov_xx = float(np.sqrt(max((A * A).mean(), 0)))
    dcov_yy = float(np.sqrt(max((B * B).mean(), 0)))
    r["distance_covariance"] = dcov_xy
    r["distance_correlation"] = _safe_div(dcov_xy, np.sqrt(dcov_xx * dcov_yy))
    return r


# ── MINE ──

def _mine_metrics(x, y):
    empty = {"MIC": np.nan, "MAS": np.nan, "MEV": np.nan, "MCN": np.nan, "MIC_minus_r2": np.nan}
    if not HAS_MINEPY or len(x) < 5:
        return empty
    try:
        mine = MINE(alpha=0.6, c=15)
        mine.compute_score(x, y)
        mic = mine.mic()
        r = pearsonr(x, y)[0] if np.std(x) > 0 and np.std(y) > 0 else np.nan
        return {
            "MIC": mic, "MAS": mine.mas(), "MEV": mine.mev(), "MCN": mine.mcn(),
            "MIC_minus_r2": mic - r ** 2 if np.isfinite(r) else np.nan,
        }
    except Exception:
        return empty


# ── Slopes ──

def _slope_metrics(x, y, prefix="raw"):
    """Endpoint + polyfit slopes for overall & 3 segments, plus segment strength."""
    keys = []
    for method in ["endpoint", "polyfit"]:
        for seg in ["overall", "early", "middle", "late"]:
            keys.append(f"{prefix}_{method}_{seg}_slope")
    keys.append(f"{prefix}_segment_strength")
    empty = {k: np.nan for k in keys}

    if len(x) < 6:
        return empty

    order = np.argsort(x)
    xs, ys = x[order], y[order]
    em, mm, lm = _segment_masks(len(xs))
    segments = {
        "overall": (xs, ys),
        "early":   (xs[em], ys[em]),
        "middle":  (xs[mm], ys[mm]),
        "late":    (xs[lm], ys[lm]),
    }

    r = {}
    ep_seg_abs = []
    for seg_name, (sx, sy) in segments.items():
        ep = _endpoint_slope(sx, sy)
        pf = _polyfit_slope(sx, sy)
        r[f"{prefix}_endpoint_{seg_name}_slope"] = ep
        r[f"{prefix}_polyfit_{seg_name}_slope"] = pf
        if seg_name != "overall":
            ep_seg_abs.append(abs(ep) if np.isfinite(ep) else np.nan)

    r[f"{prefix}_segment_strength"] = float(np.nanmean(ep_seg_abs))
    return r


def _standardized_slope_metrics(x, y):
    xn = _minmax01(x)
    yn = _minmax01(y)
    if np.any(np.isnan(xn)) or np.any(np.isnan(yn)):
        keys = []
        for method in ["endpoint", "polyfit"]:
            for seg in ["overall", "early", "middle", "late"]:
                keys.append(f"standardized_{method}_{seg}_slope")
        keys.append("standardized_segment_strength")
        return {k: np.nan for k in keys}
    return _slope_metrics(xn, yn, prefix="standardized")


# ── Bin metrics ──

def _bin_metrics(x, y, n_bins=10, bin_type="equal_width", min_count=5):
    """Amplitude, eta², buffer widths (q95−q05), n_valid_bins."""
    prefix = f"{bin_type}_bin"
    r = {
        f"{prefix}_amplitude": np.nan, f"{prefix}_eta_squared": np.nan,
        f"{prefix}_buffer_width_mean": np.nan,
        f"{prefix}_early_buffer_width": np.nan, f"{prefix}_middle_buffer_width": np.nan,
        f"{prefix}_late_buffer_width": np.nan, f"{prefix}_n_valid_bins": 0,
    }
    if len(x) < n_bins:
        return r
    try:
        bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception:
        return r
    df = pd.DataFrame({"x": x, "y": y, "bin": bins}).dropna()
    if df.empty:
        return r

    bin_stats = []
    for bid, g in df.groupby("bin", observed=True):
        if len(g) < min_count:
            continue
        yv = g["y"].values
        bw = float(np.nanpercentile(yv, 95) - np.nanpercentile(yv, 5))
        bin_stats.append({"bin": bid, "x_mean": float(g["x"].mean()),
                          "y_mean": float(yv.mean()), "buffer_width": bw, "count": len(g)})
    if len(bin_stats) < 2:
        return r

    bdf = pd.DataFrame(bin_stats).sort_values("x_mean").reset_index(drop=True)
    r[f"{prefix}_amplitude"] = float(bdf["y_mean"].max() - bdf["y_mean"].min())

    y_global = float(df["y"].mean())
    ss_tot = float(np.sum((df["y"].values - y_global) ** 2))
    ss_bet = sum(row["count"] * (row["y_mean"] - y_global) ** 2 for _, row in bdf.iterrows())
    r[f"{prefix}_eta_squared"] = _safe_div(ss_bet, ss_tot)

    r[f"{prefix}_buffer_width_mean"] = float(np.nanmean(bdf["buffer_width"]))
    r[f"{prefix}_n_valid_bins"] = len(bdf)
    nv = len(bdf)
    i1, i2 = nv // 3, 2 * nv // 3
    r[f"{prefix}_early_buffer_width"]  = float(np.nanmean(bdf.iloc[:i1]["buffer_width"]))
    r[f"{prefix}_middle_buffer_width"] = float(np.nanmean(bdf.iloc[i1:i2]["buffer_width"]))
    r[f"{prefix}_late_buffer_width"]   = float(np.nanmean(bdf.iloc[i2:]["buffer_width"]))
    return r


# ── X coverage ──

def _x_coverage_metrics(x, n_bins=10):
    r = {"x_bin_count_cv": np.nan, "x_uniform_ks_distance": np.nan}
    xf = x[np.isfinite(x)]
    if len(xf) < 3:
        return r
    lo, hi = xf.min(), xf.max()
    if hi > lo:
        xn = np.sort((xf - lo) / (hi - lo))
        n = len(xn)
        r["x_uniform_ks_distance"] = float(max(
            np.max(np.arange(1, n + 1) / n - xn),
            np.max(xn - np.arange(0, n) / n),
        ))
    if len(xf) >= n_bins:
        counts, _ = np.histogram(xf, bins=n_bins)
        mu = counts.mean()
        if mu > 0:
            r["x_bin_count_cv"] = float(np.std(counts, ddof=1) / mu)
    return r


# ── LOWESS ──

def _lowess_metrics(x, y, frac=0.25):
    empty = {
        "lowess_residual_sd": np.nan, "lowess_curve_amplitude": np.nan,
        "lowess_r2": np.nan, "lowess_first_derivative_sign_changes": np.nan,
        "lowess_overall_slope": np.nan,
    }
    if len(x) < 5:
        return empty
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    fitted = lowess(ys, xs, frac=frac, return_sorted=True)
    x_fit, y_fit = fitted[:, 0], fitted[:, 1]
    y_pred = np.interp(xs, x_fit, y_fit)
    return {
        "lowess_residual_sd": _residual_sd(ys, y_pred),
        "lowess_curve_amplitude": float(np.nanmax(y_fit) - np.nanmin(y_fit)),
        "lowess_r2": _r2(ys, y_pred),
        "lowess_first_derivative_sign_changes": _sign_changes(x_fit, y_fit),
        "lowess_overall_slope": _endpoint_slope(x_fit, y_fit),
    }

In [ ]:
def compute_per_case(x: np.ndarray, y: np.ndarray) -> dict[str, float]:
    """Core metric subset (no GAM, no equal-count bins, no y-sd normalisation)."""
    x, y = _to_valid(x, y)
    m: dict[str, float] = {}

    dm = _distance_metrics(x, y)
    m["distance_correlation"] = dm["distance_correlation"]

    m.update(_mine_metrics(x, y))
    m.update(_slope_metrics(x, y, prefix="raw"))
    m.update(_standardized_slope_metrics(x, y))
    m.update(_bin_metrics(x, y, bin_type="equal_width"))

    xcov = _x_coverage_metrics(x)
    m["x_bin_count_cv"] = xcov["x_bin_count_cv"]

    m.update(_lowess_metrics(x, y))
    m["n_valid"] = len(x)
    return m

## Compute All Metrics

In [ ]:
# Phase 1: vectorised Pearson + Spearman
t0 = time.time()
x64 = x_all.astype(np.float64)
y64 = y_all.astype(np.float64)
pearson_all = vectorised_pearson(x64, y64)
spearman_all = vectorised_spearman(x64, y64)
del x64, y64
print(f"Phase 1 done: Pearson + Spearman for {n_cases:,} cases in {time.time() - t0:.1f}s")

In [ ]:
# Phase 2: per-case metrics
REPORT_INTERVAL = 20_000

per_case_results: list[dict[str, float]] = []
t0 = time.time()

for i in tqdm(range(n_cases), desc="Per-case metrics"):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    per_case_results.append(compute_per_case(x, y))

    if (i + 1) % REPORT_INTERVAL == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (n_cases - i - 1) / rate
        print(f"  {i+1:>7,}/{n_cases:,}  ({rate:.0f} cases/s, ETA {eta/60:.1f} min)")

elapsed = time.time() - t0
print(f"\nPhase 2 done: {n_cases:,} cases in {elapsed/60:.1f} min ({n_cases / elapsed:.0f} cases/s)")

## Save Results

In [ ]:
metrics_df = pd.DataFrame(per_case_results)
metrics_df.insert(0, "case_id", cases_df["case_id"].values)
metrics_df["pearson_r"] = pearson_all
metrics_df["spearman_rho"] = spearman_all

col_order = [
    "case_id",
    # correlation / dependence
    "pearson_r", "spearman_rho", "distance_correlation",
    # MINE
    "MIC", "MAS", "MEV", "MCN", "MIC_minus_r2",
    # slopes — raw
    "raw_endpoint_overall_slope", "raw_endpoint_early_slope",
    "raw_endpoint_middle_slope", "raw_endpoint_late_slope",
    "raw_polyfit_overall_slope", "raw_polyfit_early_slope",
    "raw_polyfit_middle_slope", "raw_polyfit_late_slope",
    "raw_segment_strength",
    # slopes — standardized
    "standardized_endpoint_overall_slope", "standardized_endpoint_early_slope",
    "standardized_endpoint_middle_slope", "standardized_endpoint_late_slope",
    "standardized_polyfit_overall_slope", "standardized_polyfit_early_slope",
    "standardized_polyfit_middle_slope", "standardized_polyfit_late_slope",
    "standardized_segment_strength",
    # bins (equal-width)
    "equal_width_bin_amplitude", "equal_width_bin_eta_squared",
    "equal_width_bin_buffer_width_mean",
    "equal_width_bin_early_buffer_width", "equal_width_bin_middle_buffer_width",
    "equal_width_bin_late_buffer_width", "equal_width_bin_n_valid_bins",
    # x coverage
    "x_bin_count_cv",
    # lowess
    "lowess_residual_sd", "lowess_curve_amplitude", "lowess_r2",
    "lowess_first_derivative_sign_changes", "lowess_overall_slope",
    # misc
    "n_valid",
]

existing = [c for c in col_order if c in metrics_df.columns]
metrics_df = metrics_df[existing]

out_dir = DATA_DIR / "core"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "metrics.parquet"
metrics_df.to_parquet(out_path, index=False)
size_mb = out_path.stat().st_size / 1e6
print(f"Saved {out_path}  ({len(metrics_df):,} rows × {len(metrics_df.columns)} cols, {size_mb:.1f} MB)")
print()
display(metrics_df.head())
print()
display(metrics_df.describe().T)

## Sanity Checks

In [ ]:
check = metrics_df.merge(cases_df[["case_id", "family_id", "snr", "spread_pattern", "x_distribution"]], on="case_id")

core = ["pearson_r", "spearman_rho", "distance_correlation", "MIC",
        "equal_width_bin_eta_squared", "lowess_r2"]

null_const = (check["family_id"] == "Null") & (check["spread_pattern"] == "constant")
print("=== Null (constant spread) — expect near 0 ===")
for col in core:
    vals = check.loc[null_const, col].dropna()
    if len(vals):
        print(f"  {col:35s}  mean={vals.mean():+.4f}  |mean|={vals.abs().mean():.4f}")
print()

strong = ((check["family_id"] == "F01") & (check["snr"].astype(str) == "100")
          & (check["spread_pattern"] == "constant") & (check["x_distribution"] == "even"))
print("=== F01 Linear, SNR=100 — expect high ===")
for col in core:
    vals = check.loc[strong, col].dropna()
    if len(vals):
        print(f"  {col:35s}  mean={vals.mean():.4f}")
print()

ushape = ((check["family_id"] == "F18") & (check["snr"].astype(str) == "100")
          & (check["spread_pattern"] == "constant") & (check["x_distribution"] == "even"))
print("=== F18 U-shape, SNR=100 — |Pearson|≈0, dcor/MIC high ===")
for col in core:
    vals = check.loc[ushape, col].dropna()
    if len(vals):
        print(f"  {col:35s}  mean={vals.mean():+.4f}  |mean|={vals.abs().mean():.4f}")

print(f"\nTotal columns: {len(metrics_df.columns)}")